In [5]:
from pyspark.sql import SparkSession

In [3]:
spark = (SparkSession.builder
        .getOrCreate()
        )

print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/27 01:18:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
spark = (
    SparkSession.builder
        .appName("My app"),
        .master("local[*]"),
        .config("spark.sql.shuffle.partitions", "200"),
        .config("spark.default.parallelism", "200"),
        .getOrCreate()
)

# Spark Session
Spark entry point
    - Create application context
    - Manage configurations
    - Allow access to SQL, Dataframes, Catalogs, read/write

`spark = SparkSession.builder.getOrCreate()` creates:
    - SparkContext
    - SQL Context
    - SessionState
    - Catalog
    - RuntimeConfig


## Data Reading
 df = spark.read.parquet("s3://bucket/path")
 df = spark.read.json("file.json")
 df = spark.read.format("jdbc").load()

 Spark session will:
     - Chose a DataSource
     - Reading Parallelism
     - Infer Schema (or not)

## Data Write
 df.write.mode("overwrite").partitionBy("updated_at").parquet(path)

 Spark Session will control:
    - number of files
    - compression
    - partition
    - format

In [16]:
!ls /data
!ls /data/archive

archive
taxi+_zone_lookup.csv	     yellow_tripdata_2019-09.csv
taxi_zones		     yellow_tripdata_2019-10.csv
yellow_tripdata_2019-01.csv  yellow_tripdata_2019-11.csv
yellow_tripdata_2019-02.csv  yellow_tripdata_2019-12.csv
yellow_tripdata_2019-03.csv  yellow_tripdata_2020-01.csv
yellow_tripdata_2019-04.csv  yellow_tripdata_2020-02.csv
yellow_tripdata_2019-05.csv  yellow_tripdata_2020-03.csv
yellow_tripdata_2019-06.csv  yellow_tripdata_2020-04.csv
yellow_tripdata_2019-07.csv  yellow_tripdata_2020-05.csv
yellow_tripdata_2019-08.csv  yellow_tripdata_2020-06.csv


In [15]:
spark = (
    SparkSession.builder
        .appName("testing")
        .master("spark://spark-master:7077")
        .getOrCreate()
)

df = spark.read.option("header", True).csv("/data/archive/yellow_tripdata_2019-01.csv")
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|         1.50|         1|                 N|         151|         239|           1|          7|  0.5|    0.5|      1.65|           0|                  0.3

Spark Application doesnt run locally, it ran in a cluster
The code ran insede a driver(Jupyter in this case), but the data are processed in workers
Anything that the workers dont see (path, lib, permission) it will break

Driver is different from worker:
    - when i did `!ls /data`
    - This doesnt guarantee that the worker will see the file. So, that why the volume `/data` need to be mounted in all containers and with the same path. 

Path on Spark IS ALWAYS from the cluster
 `csv("data/archive/file.csv")` - wrong
  `csv("/data/archive/file.csv")` - wrong

Spark dont use:
    - host filesystem or notebook filesystem
Spark use:
    - HDFS
    - S3
    - GCS
    - Docker Volume
    - NFS

Spark Session is not only for reading, its also: 
    Reading: 
    spark.read.csv(...)
    spark.read.parquet(...)
    spark.read.json(...)
    spark.read.table("db.table")

    Writing:
    df.write.parquet(...)
    df.write.mode("overwrite").saveAsTable(...)

    Performance Configuration:
    spark.conf.set("spark.sql.shuffle.partitions", 8)
    spark.conf.get("spark.executor.memory")

    Lazy evaluation
    df.show() - Action
    